In [ ]:
import math

# Configurações da Imagem
LARGURA = 400
ALTURA = 300
VALOR_MAX = 255
CAMINHO_SAIDA = "imagem_gerada.ppm"


def gerar_imagem_ppm():
    dados_pixels = bytearray()

    # Parâmetros das formas geométricas
    # 1. Círculo (Centro X, Centro Y, Raio)
    circulo_x, circulo_y, circulo_raio = 120, 150, 70

    # 2. Retângulo (X_min, Y_min, X_max, Y_max)
    ret_x1, ret_y1, ret_x2, ret_y2 = 230, 80, 350, 220

    print("Gerando pixels da imagem...")

    for y in range(ALTURA):
        for x in range(LARGURA):

            # --- DESENHO DO CÍRCULO ---
            # Equação do círculo: (x - cx)^2 + (y - cy)^2 <= r^2
            distancia_ao_centro = math.sqrt((x - circulo_x) ** 2 + (y - circulo_y) ** 2)

            # --- VERIFICAÇÃO DAS FORMAS ---
            if distancia_ao_centro <= circulo_raio:
                # Cor do Círculo: Amarelo (255, 215, 0)
                r, g, b = 255, 215, 0

            elif ret_x1 <= x <= ret_x2 and ret_y1 <= y <= ret_y2:
                # Cor do Retângulo: Azul Cian (0, 180, 216)
                r, g, b = 0, 180, 216

            else:
                # Fundo: Degradê suave de Roxo/Azul para Laranja/Vermelho
                r = int(255 * (x / LARGURA))
                g = int(100 * (y / ALTURA))
                b = int(255 * (1 - x / LARGURA))

            # Adiciona os 3 bytes de cor (RGB) para cada pixel
            dados_pixels.append(r)
            dados_pixels.append(g)
            dados_pixels.append(b)

    # --- SALVANDO NO FORMATO PPM P6 ---
    print(f"Escrevendo arquivo: {CAMINHO_SAIDA}...")
    with open(CAMINHO_SAIDA, "wb") as f:
        # 1. Cabeçalho PPM P6
        cabecalho = f"P6\n{LARGURA} {ALTURA}\n{VALOR_MAX}\n".encode("ascii")
        f.write(cabecalho)

        # 2. Bloco binário de dados RGB
        f.write(dados_pixels)

    print("Imagem gerada com sucesso!")


if __name__ == "__main__":
    gerar_imagem_ppm()

Gerando pixels da imagem...
Escrevendo arquivo: imagem_gerada.ppm...
Imagem gerada com sucesso!


In [ ]:
# Caminho da imagem de entrada no formato PPM (P6 - Binário)
CAMINHO_IMAGEM_ORIGINAL = "imagem_gerada.ppm"

# Limiar (threshold) para binarização (0 a 255)
LIMIAR_BINARIZACAO = 128


def ler_ppm_p6(caminho_arquivo):
    """
    Lê uma imagem no formato PPM (P6 - Binário) sem usar bibliotecas.
    Retorna o cabeçalho (largura, altura, valor_max) e os dados brutos de pixels (RGB).
    """
    with open(caminho_arquivo, "rb") as f:
        # Lê a assinatura do arquivo (Magic Number - P6)
        linha = f.readline().strip()
        if linha != b"P6":
            raise ValueError("O arquivo deve estar no formato PPM P6 (binário).")

        # Ignora comentários iniciados por '#'
        linha = f.readline()
        while linha.startswith(b"#"):
            linha = f.readline()

        # Lê largura e altura
        dimensoes = linha.split()
        largura = int(dimensoes[0])
        altura = int(dimensoes[1])

        # Lê o valor máximo de cor (geralmente 255)
        valor_max = int(f.readline().strip())

        # Lê os dados de pixels restantes (RGB - 3 bytes por pixel)
        dados_pixels = bytearray(f.read())

    return largura, altura, valor_max, dados_pixels


def salvar_pgm_p5(caminho_saida, largura, altura, valor_max, dados_cinza):
    """
    Salva uma imagem em escala de cinza no formato PGM (P5 - Binário).
    """
    with open(caminho_saida, "wb") as f:
        cabecalho = f"P5\n{largura} {altura}\n{valor_max}\n".encode("ascii")
        f.write(cabecalho)
        f.write(dados_cinza)


def processar_imagem():
    print(f"Lendo a imagem de origem: {CAMINHO_IMAGEM_ORIGINAL}...")

    # 1. Leitura do arquivo PPM
    largura, altura, valor_max, dados_rgb = ler_ppm_p6(CAMINHO_IMAGEM_ORIGINAL)

    pixels_cinza = bytearray()
    pixels_binarios = bytearray()

    total_pixels = largura * altura

    # 2. Conversão pixel a pixel
    for i in range(total_pixels):
        # Cada pixel possui 3 bytes: R, G, B
        r = dados_rgb[i * 3]
        g = dados_rgb[i * 3 + 1]
        b = dados_rgb[i * 3 + 2]

        # Fórmula de luminância NTSC/PAL para conversão em escala de cinza:
        # Y = 0.299*R + 0.587*G + 0.114*B
        tom_cinza = int(0.299 * r + 0.587 * g + 0.114 * b)
        pixels_cinza.append(tom_cinza)

        # Binarização com base no limiar (Threshold)
        if tom_cinza >= LIMIAR_BINARIZACAO:
            pixels_binarios.append(255)  # Branco
        else:
            pixels_binarios.append(0)    # Preto

    # 3. Salvar imagem em níveis de cinza (PGM P5)
    caminho_cinza = "imagem_gerada.pgm"
    salvar_pgm_p5(caminho_cinza, largura, altura, valor_max, pixels_cinza)
    print(f"Imagem em escala de cinza salva em: {caminho_cinza}")

    # 4. Salvar imagem binarizada (PBM/PGM P5 com 0 e 255)
    caminho_binaria = "imagem_gerada_2.pgm"
    salvar_pgm_p5(caminho_binaria, largura, altura, valor_max, pixels_binarios)
    print(f"Imagem binarizada (preto e branco) salva em: {caminho_binaria}")


if __name__ == "__main__":
    processar_imagem()

Lendo a imagem de origem: imagem_gerada.ppm...
Imagem em escala de cinza salva em: imagem_gerada.pgm
Imagem binarizada (preto e branco) salva em: imagem_gerada_2.pgm
